# Step 2: Risk Stratification (Charlson Comorbidity Index)

In healthcare analytics, **Risk Stratification** is the process of assigning a risk score to patients to predict future healthcare usage (like hospital readmissions).

One of the most common methods is the **Charlson Comorbidity Index (CCI)**. It assigns points to patients based on their diagnosis history. Higher scores = sicker patients.

In this notebook, we will:
1. Define a simplified scoring rule based on ICD-10 codes.
2. Calculate a Risk Score for every patient in our dataset.
3. Segment patients into "Low Risk", "Medium Risk", and "High Risk" cohorts.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

# Load Data
DATA_DIR = '../data'
df_patients = pd.read_csv(os.path.join(DATA_DIR, 'patients.csv'))
df_diagnoses = pd.read_csv(os.path.join(DATA_DIR, 'diagnoses.csv'))

print("Data loaded.")

### 1. Define Risk Weights
We will map specific ICD-10 codes to a severity score (weight). In the real world, this mapping table is thousands of lines long. We will use a simplified version for our specific dataset.

In [ ]:
# Dictionary mapping ICD-10 prefixes/codes to Charlson Weights
# Examples based on standard CCI logic:
# Diabetes = 1 point
# COPD/Asthma = 1 point
# Hypertension (added for demo) = 1 point

CCI_WEIGHTS = {
    'E11': 1,   # Diabetes
    'J45': 1,   # Asthma (Chronic Pulmonary Disease proxy)
    'I10': 1,   # Hypertension
    'M54': 0,   # Back pain (usually 0 risk)
    'Z00': 0,   # General Exam (0 risk)
    'J06': 0,   # Acute infection (temporary, 0 risk)
    'E78': 0    # Hyperlipidemia (0 in standard CCI, though important)
}

def get_weight(icd_code):
    # Check the first 3 characters of the code (ICD Category)
    prefix = icd_code[:3]
    return CCI_WEIGHTS.get(prefix, 0)

# Test the function
print(f"Weight for E11.9 (Diabetes): {get_weight('E11.9')}")
print(f"Weight for Z00.00 (General Exam): {get_weight('Z00.00')}")

### 2. Calculate Score per Patient
A patient's score is the SUM of weights for all their *unique* chronic conditions.

In [ ]:
# 1. Apply weights to the diagnosis table
df_diagnoses['risk_points'] = df_diagnoses['icd_code'].apply(get_weight)

# 2. We only want to count each condition ONCE per patient.
# (If you have Diabetes diagnosed 5 times, you still only get 1 point)
df_unique_conditions = df_diagnoses[['patient_id', 'icd_code', 'risk_points']].drop_duplicates()

# 3. Sum the points by patient
patient_risk_scores = df_unique_conditions.groupby('patient_id')['risk_points'].sum().reset_index()
patient_risk_scores.rename(columns={'risk_points': 'total_risk_score'}, inplace=True)

# 4. Merge scores back to the main patient list (to include patients with 0 score)
df_scored = pd.merge(df_patients, patient_risk_scores, on='patient_id', how='left')
df_scored['total_risk_score'] = df_scored['total_risk_score'].fillna(0) # Fill NaNs with 0

df_scored.head(10)

### 3. Analyze the Population Risk
Now we can see the distribution of sickness in our population.

In [ ]:
print(df_scored['total_risk_score'].value_counts().sort_index())

### 4. Create Risk Segments
For reporting, we often group scores into buckets:
- 0: Healthy
- 1: Moderate Risk
- 2+: High Risk (These patients need Care Management)

In [ ]:
def classify_risk(score):
    if score == 0: return 'Healthy'
    if score == 1: return 'Moderate'
    return 'High Risk'

df_scored['risk_segment'] = df_scored['total_risk_score'].apply(classify_risk)

# View counts per segment
df_scored['risk_segment'].value_counts().plot(kind='bar', color=['green', 'orange', 'red'])
plt.title('Patient Population by Risk Level')
plt.xlabel('Risk Segment')
plt.ylabel('Number of Patients')
plt.show()

### Save the Results
We will save this scored dataset for use in our Dashboard later.

In [ ]:
df_scored.to_csv(os.path.join(DATA_DIR, 'patients_with_scores.csv'), index=False)
print("Saved scored patient list.")